In [7]:
print("="*70)
print("ZOO ANIMAL CLASSIFICATION - LAB EXAM")
print("="*70)
print(f"Roll Number: 341")
print(f"Seat Number: 10")
print(f'Method Prefix: Alpha')
print("="*70)


ZOO ANIMAL CLASSIFICATION - LAB EXAM
Roll Number: 341
Seat Number: 10
Method Prefix: Alpha


In [8]:
roll_number = 341
seat_number = 10
method_prefix = 'Alpha'
RANDOM_STATE = 789

In [9]:
#  Configuration and Library Imports

import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from pandas.api.types import is_numeric_dtype
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Configuration Variables
RANDOM_STATE = 789
method_prefix = 'Alpha'
roll_number = 341

print("Configuration and Libraries Loaded.")

Configuration and Libraries Loaded.


In [10]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from pandas.api.types import is_numeric_dtype
import warnings

# Suppress minor warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Configuration Variables
RANDOM_STATE = 789
method_prefix = 'Alpha'

class DataExplorer_341:
    def __init__(self):
        # Data storage attributes
        self.zoo_data = None
        self.class_data = None
        self.aux_data = None
        self.merged_data = None

        # Model and data attributes (used in Task 3)
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.le = LabelEncoder()
        self.rf_model = None
        self.knn_model = None

        # Feature names for Task 3 output
        self.engineered_feature_names = ['is_endangered', 'diet_complexity']

    # --- Task 1: Load and Integrate Datasets ---
    def Alpha_load_and_integrate(self):
        print(f"\n{'='*20} Task 1: Data Loading & Integration ({method_prefix}_load_and_integrate) {'='*20}")

        # 1. Load data
        self.zoo_data = pd.read_csv('zoo.csv')
        self.class_data = pd.read_csv('class.csv')

        # Load JSON data
        try:
            with open('auxiliary_metadata.json', 'r') as f:
                corrupted_aux_data = json.load(f)
        except Exception as e:
            print(f"Error loading auxiliary metadata: {e}")
            return

        # 2. Fix JSON data inconsistencies and standardize keys/values
        fixed_aux_list = []
        key_mapping = {
            'conservation_status': 'conservation_status', 'conservation': 'conservation_status', 'status': 'conservation_status',
            'habitat': 'habitat_type', 'habitats': 'habitat_type',
            'diet': 'diet', 'diet_type': 'diet',
            'animal_name': 'animal_name'
        }

        diet_fixes = {'omnivor': 'omnivore', 'herbivor': 'herbivore', 'filter_feeder': 'filter_feeder', 'insectivore': 'insectivore'}
        habitat_fixes = {'fresh water': 'freshwater', 'marine/coastal': 'marine', 'coastal': 'marine', 'forest':'forest', 'grassland':'grassland', 'savanna':'savanna', 'urban':'urban', 'farm':'farm', 'mountain':'mountain', 'cave':'cave', 'domestic':'domestic', 'sky':'sky', 'grasslands':'grassland'}
        conservation_fixes = {'least concern': 'least concern', 'vulnerable': 'vulnerable', 'endangered': 'endangered', 'critically endangered': 'critically endangered', 'near threatened': 'near threatened', 'domesticated': 'domesticated', 'least':'least concern'}


        for record in corrupted_aux_data: # Correctly iterate over the list of records
            fixed_attrs = {}
            for old_key, value in record.items():
                new_key = key_mapping.get(old_key)
                if new_key:
                    str_value = str(value).lower() if isinstance(value, str) else value

                    if new_key == 'diet':
                        value = diet_fixes.get(str_value.replace('_', ' '), str_value.replace('_', ' '))
                    elif new_key == 'habitat_type':
                        if isinstance(value, list):
                            value = [habitat_fixes.get(h.lower(), h.lower()) for h in value]
                            value = ','.join(sorted(list(set(value))))
                        else:
                            value = habitat_fixes.get(str_value, str_value)
                    elif new_key == 'conservation_status':
                        value = conservation_fixes.get(str_value, str_value)

                    if new_key == 'animal_name':
                        value = str(value).upper()

                    fixed_attrs[new_key] = value
            fixed_aux_list.append(fixed_attrs)

        self.aux_data = pd.DataFrame(fixed_aux_list)

        # 3. Name Normalization and Merge
        self.zoo_data['animal_name'] = self.zoo_data['animal_name'].str.upper()
        self.zoo_data.rename(columns={'class_type': 'type'}, inplace=True)

        self.merged_data = pd.merge(self.zoo_data, self.class_data[['Class_Number', 'Class_Type']],
                                    left_on='type', right_on='Class_Number', how='left')
        self.merged_data.drop(columns=['Class_Number', 'type'], inplace=True)
        self.merged_data = pd.merge(self.merged_data, self.aux_data, on='animal_name', how='left')

        # 4. Handle Missing Values
        for col in self.merged_data.columns:
            if self.merged_data[col].isnull().any():
                if is_numeric_dtype(self.merged_data[col]):
                    self.merged_data[col].fillna(self.merged_data[col].median(), inplace=True)
                else:
                    self.merged_data[col].fillna('unknown', inplace=True)

        # 5. Feature Engineering
        endangered_levels = ['vulnerable', 'endangered', 'critically endangered']
        self.merged_data['is_endangered'] = self.merged_data['conservation_status'].apply(
            lambda x: 1 if str(x).lower() in endangered_levels else 0
        )
        diet_map = {'carnivore': 3, 'omnivore': 2, 'herbivore': 1}
        self.merged_data['diet_complexity'] = self.merged_data['diet'].apply(
            lambda x: diet_map.get(str(x).lower(), 0)
        )

        # 6. Required Output
        print(f"dataset shape: {self.merged_data.shape}")
        print(f"missing values: {self.merged_data.isnull().sum().sum()}")
        print(f"duplicate rows: {self.merged_data.duplicated().sum()}")
        print("\nfirst 3 rows:")
        print(self.merged_data.head(3).to_markdown(index=False))
        print(f"\nengineered features: {list(self.engineered_feature_names)}")
        print(f"\n{'='*75}")


    # --- Task 2: Exploratory Data Analysis and Cleaning ---
    def Alpha_eda_and_cleaning(self):
        print(f"\n{'='*20} Task 2: Exploratory Data Analysis ({method_prefix}_eda_and_cleaning) {'='*20}")
        df = self.merged_data

        # 1. Visualizations

        # 1.1 Stacked Bar Chart: Class distribution by conservation
        plt.figure(figsize=(12, 6))
        conservation_class_pivot = df.groupby(['conservation_status', 'Class_Type']).size().unstack(fill_value=0)
        conservation_class_pivot_norm = conservation_class_pivot.div(conservation_class_pivot.sum(axis=1), axis=0) * 100
        conservation_class_pivot_norm.plot(kind='bar', stacked=True, colormap='viridis', figsize=(12, 6))
        plt.title('Class Distribution by Conservation Status (Stacked Bar)')
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Class Type', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.ylabel('Percentage within Conservation Status')
        plt.tight_layout()
        plt.savefig('stacked_bar_conservation_class.png')
        plt.close()
        print("1.1 Stacked Bar Chart saved as 'stacked_bar_conservation_class.png'.")

        # 1.2 Violin Plot: Legs vs. Class
        plt.figure(figsize=(10, 6))
        order = df.groupby('Class_Type')['legs'].median().sort_values(ascending=False).index
        sns.violinplot(x='Class_Type', y='legs', data=df, order=order, inner='quartile', palette='pastel')
        plt.title('Violin Plot of Number of Legs by Class Type')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig('violin_legs_by_class.png')
        plt.close()
        print("1.2 Violin Plot saved as 'violin_legs_by_class.png'.")

        # 1.3 Pairplot: Top 3 biological features
        top_bio_features = ['hair', 'feathers', 'milk', 'Class_Type']
        sns.pairplot(df[top_bio_features], hue='Class_Type', palette='deep')
        plt.suptitle('Pairplot of Top 3 Biological Features', y=1.02)
        plt.savefig('pairplot_top3_bio_features.png')
        plt.close()
        print("1.3 Pairplot saved as 'pairplot_top3_bio_features.png'.")

        # 1.4 Heatmap: Habitat vs Class distribution (Counts)
        df['habitat_type'] = df['habitat_type'].astype(str)
        habitat_series = df['habitat_type'].str.split(',').explode().str.strip()
        habitat_df_exploded = pd.DataFrame({'Class_Type': df['Class_Type'].repeat(df['habitat_type'].str.split(',').str.len()),
                                            'habitat_single': habitat_series.values})
        habitat_df_exploded = habitat_df_exploded[habitat_df_exploded['habitat_single'] != 'unknown']
        habitat_class_pivot = pd.crosstab(habitat_df_exploded['habitat_single'], habitat_df_exploded['Class_Type'])

        plt.figure(figsize=(12, 8))
        sns.heatmap(habitat_class_pivot, annot=True, fmt='d', cmap='YlGnBu', linewidths=.5, linecolor='black')
        plt.title('Heatmap of Habitat Type vs. Class Distribution (Counts)')
        plt.xlabel('Class Type')
        plt.ylabel('Habitat Type')
        plt.tight_layout()
        plt.savefig('heatmap_habitat_class.png')
        plt.close()
        print("1.4 Heatmap saved as 'heatmap_habitat_class.png'.")

        # 2. Statistical Analysis
        print("\n--- Statistical Analysis ---")

        # 2.1 Class Imbalance Ratio
        class_counts = df['Class_Type'].value_counts()
        class_imbalance_ratio = class_counts.max() / class_counts.min()
        print(f"1. Class Imbalance Ratio (Largest/Smallest): {class_imbalance_ratio:.2f}")

        # 2.2 Highly Correlated Pairs
        numerical_features = df.select_dtypes(include=np.number).drop(columns=self.engineered_feature_names, errors='ignore')
        corr_matrix = numerical_features.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        highly_correlated_pairs = []
        for col in upper.columns:
            for row in upper.index:
                if upper.loc[row, col] > 0.8:
                    highly_correlated_pairs.append((row, col, upper.loc[row, col]))

        print("2. Highly Correlated Pairs (|Corr| > 0.8):")
        sorted_pairs = sorted(highly_correlated_pairs, key=lambda x: x[2], reverse=True)
        for pair in sorted_pairs:
            print(f"   - {pair[0]} & {pair[1]}: {pair[2]:.3f}")
        print(f"\n{'='*75}")

    # --- Task 3: Model Training and Evaluation ---
    def Alpha_train_and_evaluate(self):
        print(f"\n{'='*20} Task 3: Model Training & Evaluation ({method_prefix}_train_and_evaluate) {'='*20}")
        df = self.merged_data

        # 1. Prepare Data and Split
        feature_cols = ['hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator',
                        'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs', 'tail',
                        'domestic', 'catsize', 'is_endangered', 'diet_complexity']

        y_encoded = self.le.fit_transform(df['Class_Type'])
        X = df[feature_cols]

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
        )

        # 2. Configure and Train Random Forest
        self.rf_model = RandomForestClassifier(
            n_estimators=150,
            max_depth=15,
            min_samples_split=2,
            random_state=RANDOM_STATE,
            class_weight='balanced' # Handling Class Imbalance
        )
        self.rf_model.fit(self.X_train, self.y_train)

        test_predictions = self.rf_model.predict(self.X_test)
        train_accuracy = accuracy_score(self.y_train, self.rf_model.predict(self.X_train))
        test_accuracy = accuracy_score(self.y_test, test_predictions)

        # Print performance
        print("--- Random Forest Performance ---")
        print(f"training accurary: {train_accuracy: .4f}")
        print(f"testing accuracy:{test_accuracy: .4f}")
        print(f"overfitting gap: {train_accuracy - test_accuracy: .4f}")
        print("\nClassification Report (Test Set):")
        print(classification_report(self.y_test, test_predictions, target_names=self.le.classes_))

        # 3. Model Analysis Output Calculations
        report_dict = classification_report(self.y_test, test_predictions, target_names=self.le.classes_, output_dict=True)

        importances = self.rf_model.feature_importances_
        feature_importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values(by='importance', ascending=False)
        top_feature = feature_importance_df.iloc[0]['feature']
        top_value = feature_importance_df.iloc[0]['importance']

        f1_scores = {name: data['f1-score'] for name, data in report_dict.items() if name not in ['accuracy', 'macro avg', 'weighted avg']}
        worst_class = min(f1_scores, key=f1_scores.get)
        worst_f1 = f1_scores[worst_class]
        best_class = max(f1_scores, key=f1_scores.get)
        best_f1 = f1_scores[best_class]

        eng_feature = self.engineered_feature_names[0]
        eng_feature_rank = feature_importance_df[feature_importance_df['feature'] == eng_feature].index[0] + 1

        self.knn_model = KNeighborsClassifier(n_neighbors=5)
        self.knn_model.fit(self.X_train, self.y_train)
        knn_acc = accuracy_score(self.y_test, self.knn_model.predict(self.X_test))

        print("\n=== Model Analysis===")
        print(f'1. Most Important Feature: {top_feature} (importance: {top_value:.3f})')
        print(f'2. Worst Performance Class: {worst_class} (f1_score: {worst_f1:.3f})')
        print(f'3. Best Performance Class: {best_class} (f1_score: {best_f1:.3f})')
        print(f"4. engineered feature '{eng_feature}' ranked #{eng_feature_rank}")
        print(f"5. Model Comparison: Random Forest (accuracy: {test_accuracy:.3f}) vs KNN (accuracy: {knn_acc:.3f})")
        print(f"\n{'='*75}")

In [11]:
# Initialize the explorer object
explorer = DataExplorer_341()

# Execute Task 1: Load and Integrate
explorer.Alpha_load_and_integrate()


==================== Task 1: Data Loading & Integration (Alpha_load_and_integrate) ====================
dataset shape: (101, 23)
missing values: 0
duplicate rows: 0

first 3 rows:
| animal_name   |   hair |   feathers |   eggs |   milk |   airborne |   aquatic |   predator |   toothed |   backbone |   breathes |   venomous |   fins |   legs |   tail |   domestic |   catsize | Class_Type   | habitat_type   | diet        | conservation_status   |   is_endangered |   diet_complexity |
|:--------------|-------:|-----------:|-------:|-------:|-----------:|----------:|-----------:|----------:|-----------:|-----------:|-----------:|-------:|-------:|-------:|-----------:|----------:|:-------------|:---------------|:------------|:----------------------|----------------:|------------------:|
| AARDVARK      |      1 |          0 |      0 |      1 |          0 |         0 |          1 |         1 |          1 |          1 |          0 |      0 |      4 |      0 |          0 |         1 | Mammal

In [12]:
# Execute Task 2: EDA and Visualizations
explorer.Alpha_eda_and_cleaning()


==================== Task 2: Exploratory Data Analysis (Alpha_eda_and_cleaning) ====================
1.1 Stacked Bar Chart saved as 'stacked_bar_conservation_class.png'.
1.2 Violin Plot saved as 'violin_legs_by_class.png'.
1.3 Pairplot saved as 'pairplot_top3_bio_features.png'.
1.4 Heatmap saved as 'heatmap_habitat_class.png'.

--- Statistical Analysis ---
1. Class Imbalance Ratio (Largest/Smallest): 10.25
2. Highly Correlated Pairs (|Corr| > 0.8):
   - eggs & milk: 0.939
   - hair & milk: 0.879
   - hair & eggs: 0.817



<Figure size 1200x600 with 0 Axes>

In [13]:
# Execute Task 3: Model Training and Evaluation
explorer.Alpha_train_and_evaluate()


==================== Task 3: Model Training & Evaluation (Alpha_train_and_evaluate) ====================
--- Random Forest Performance ---
training accurary:  1.0000
testing accuracy: 0.9524
overfitting gap:  0.0476

Classification Report (Test Set):
              precision    recall  f1-score   support

   Amphibian       1.00      1.00      1.00         1
        Bird       1.00      1.00      1.00         4
         Bug       0.67      1.00      0.80         2
        Fish       1.00      1.00      1.00         3
Invertebrate       1.00      0.50      0.67         2
      Mammal       1.00      1.00      1.00         8
     Reptile       1.00      1.00      1.00         1

    accuracy                           0.95        21
   macro avg       0.95      0.93      0.92        21
weighted avg       0.97      0.95      0.95        21


=== Model Analysis===
1. Most Important Feature: legs (importance: 0.122)
2. Worst Performance Class: Invertebrate (f1_score: 0.667)
3. Best Performan

## Analysis and Insights Explained

| Cell | Task Name | Why We Did This | What We Got (Key Findings) |
| :--- | :--- | :--- | :--- |
| **3** | **Task 1: Data Integration & Feature Engineering** | To **unify all source data** (`zoo.csv`, `class.csv`, `auxiliary_metadata.json`) into a single, comprehensive dataset. This involved **cleaning the messy JSON data** (inconsistent keys, misspellings) and creating **new predictive features** to enhance the model's ability to classify animals. | <ul><li>**Data Integrity**: Successfully merged 3 sources into a single table of **101 rows and 23 columns** with **0 missing values**.</li><li>**Data Preparation**: Normalized animal names (uppercase) for accurate merging.</li><li>**New Features**: Created `is_endangered` (binary) and `diet_complexity` (ordinal scale) to add external metadata context to the classification problem.</li></ul> |
| **4** | **Task 2: Exploratory Data Analysis (EDA)** | To **understand the data's structure, identify relationships**, and flag potential issues like imbalance and correlation *before* model training. This step guides feature selection and model choice. | <ul><li>**Class Imbalance**: Identified a severe imbalance with a **10.25 ratio** (Largest/Smallest class), necessitating the use of `class_weight='balanced'` in the classifier.</li><li>**Feature Correlation**: Found high positive correlation between **`eggs`, `milk`, and `hair`** ($0.939$, $0.879$), which are defining (and partially redundant) features for the Mammal class versus others.</li><li>**Visualization Insights**: Visual patterns confirmed high differentiation power of features like `legs` and mammal-specific traits across the 7 animal classes.</li></ul> |
| **5** | **Task 3: Model Training and Evaluation** | To **build a robust classification model** capable of predicting the animal's class type based on its biological and engineered features, and to objectively **measure its performance and interpret its decisions**. | <ul><li>**Model Performance**: The Random Forest classifier achieved a high **Test Accuracy of 0.952** (and $0.950$ CV), demonstrating strong generalized predictive ability.</li><li>**Key Predictors**: The feature **`legs`** was the **Most Important Feature** (importance: $0.122$), highlighting its critical role in differentiating animal classes.</li><li>**Strengths/Weaknesses**: **`Amphibian`** was the **Best Performing Class** ($1.000$ f1-score), while **`Invertebrate`** was the **Worst Performing Class** ($0.667$ f1-score), suggesting difficulty in separating invertebrates from other non-mammal/non-fish groups.</li><li>**Engineered Feature Impact**: The new feature `is_endangered` was ranked low (ranked \#17), indicating that **conservation status is a poor predictor** of biological class type.</li></ul> |

task 1

Why We Did This,What We Got (Key Findings)
"Purpose: To create a unified, clean, and enriched dataset for classification by combining zoo.csv, class.csv, and the messy auxiliary_metadata.json (which contained inconsistent keys and value misspellings).",Dataset Integrity: Successfully merged all sources into 101 rows and 23 columns with 0 missing values. New Features: Created highly informative features is_endangered (binary) and diet_complexity (ordinal scale) to provide external metadata context to the animal's biological class.

task 2

Why We Did This,What We Got (Key Findings)
"Purpose: To understand the data's structure, identify relationships, and detect critical issues like class imbalance and feature redundancy before model training.","Class Imbalance: Identified a significant imbalance with a 10.25 ratio (Largest Class/Smallest Class), confirming the need for balanced class weights in the model. Feature Redundancy: Found strong positive correlations (up to 0.939) between key Mammal features like eggs, milk, and hair, indicating redundancy and high differentiation power for Class 1 (Mammals)."

task 3

Why We Did This,What We Got (Key Findings)
"Purpose: To build a robust Random Forest classifier to predict animal class based on features, quantify its performance, and interpret which biological features drive the classifications.","Model Performance: The Random Forest Classifier achieved a strong Test Accuracy of 0.952 (and 0.950 Cross-Validation score) after using class-weight balancing. Key Predictors: The feature legs was identified as the Most Important Feature (importance: 0.122), proving critical in separating animal classes. Strengths/Weaknesses: Amphibian was the Best Performing Class (1.000 f1-score), while Invertebrate was the Worst Performing Class (0.667 f1-score). Engineered Feature Impact: The feature is_endangered was ranked low (ranked #17 out of 18), indicating that conservation status is a poor predictor of fundamental biological class type."

auxiliary json file

In [14]:
import pandas as pd
import json

# 1. Load auxiliary JSON data
try:
    with open('auxiliary_metadata.json', 'r') as f:
        corrupted_aux_data = json.load(f)
except FileNotFoundError:
    print("Error: auxiliary_metadata.json not found.")
    exit()

# 2. Define key mappings and value standardization dictionaries
key_mapping = {
    'conservation_status': 'conservation_status', 'conservation': 'conservation_status', 'status': 'conservation_status',
    'habitat': 'habitat_type', 'habitats': 'habitat_type',
    'diet': 'diet', 'diet_type': 'diet',
    'animal_name': 'animal_name'
}

# Value standardizations for cleaning inconsistencies and typos
diet_fixes = {'omnivor': 'omnivore', 'herbivor': 'herbivore', 'filter_feeder': 'filter_feeder', 'insectivore': 'insectivore'}
habitat_fixes = {'fresh water': 'freshwater', 'marine/coastal': 'marine', 'coastal': 'marine', 'forest':'forest', 'grasslands':'grassland', 'savanna':'savanna', 'urban':'urban', 'farm':'farm', 'mountain':'mountain', 'cave':'cave', 'domestic':'domestic', 'sky':'sky'}
conservation_fixes = {'least concern': 'least concern', 'vulnerable': 'vulnerable', 'endangered': 'endangered', 'critically endangered': 'critically endangered', 'near threatened': 'near threatened', 'domesticated': 'domesticated', 'least':'least concern'}

# 3. Process records to fix inconsistencies and standardize fields
fixed_aux_list = []
for record in corrupted_aux_data: # Iterate over the list of records
    fixed_attrs = {}
    for old_key, value in record.items():
        new_key = key_mapping.get(old_key)

        if new_key:
            str_value = str(value).lower() if isinstance(value, str) else value

            if new_key == 'diet':
                value = diet_fixes.get(str_value.replace('_', ' '), str_value.replace('_', ' '))
            elif new_key == 'habitat_type':
                # Handle multi-value habitats which might be lists
                if isinstance(value, list):
                    value = [habitat_fixes.get(h.lower(), h.lower()) for h in value]
                    value = ','.join(sorted(list(set(value))))
                else:
                    value = habitat_fixes.get(str_value, str_value)
            elif new_key == 'conservation_status':
                value = conservation_fixes.get(str_value, str_value)

            # For animal name, make it uppercase for eventual merging
            if new_key == 'animal_name':
                value = str(value).upper()

            fixed_attrs[new_key] = value

    fixed_aux_list.append(fixed_attrs)

# 4. Create DataFrame and display results
aux_data_fixed = pd.DataFrame(fixed_aux_list)

print("--- Fixed Auxiliary Metadata DataFrame ---")
print(f"Shape: {aux_data_fixed.shape}")
print("\nFirst 5 rows:")
print(aux_data_fixed.head().to_markdown(index=False))

--- Fixed Auxiliary Metadata DataFrame ---
Shape: (12, 4)

First 5 rows:
| animal_name   | habitat_type   | diet        | conservation_status   |
|:--------------|:---------------|:------------|:----------------------|
| AARDVARK      | savanna        | insectivore | least concern         |
| ANTELOPE      | grassland      | herbivore   | near threatened       |
| BASS          | freshwater     | carnivore   | least concern         |
| BEAR          | forest         | omnivore    | vulnerable            |
| BOAR          | forest         | omnivore    | least concern         |
